# ACE energy against the network, bin by bin

The validation loss of an ACE energy (a linear fit of `fit.py` or a trained embedding of `embedding.py`) and of a network
on the same draws as `fit.py` (5000 validation poses, one noise draw per pose), split by bins of the noise level; the
worst poses of a bin; for one pose, the error per joint, the energy's score split by feature group, the energy per joint
and the skeleton. Report: `docs/experiments/ace/README.md`.

Run from `explore/ace_energy/` (or anywhere: the helpers find the repository).

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

import ipywidgets as w
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / "explore" / "ace_energy"))
from inspect_energy import (FOLDER, PARENTS, decompose, energy_per_joint, energy_predictions, load_energy,
                            network_predictions, points, validation_draws)
from hp.data.amass_cfg import AmassJoints4Learning
from hp.models.io import load_model

JOINTS = [j.name for j in AmassJoints4Learning]
NETWORK = "~/data/experiments/factorial/rsgm_posedflow_fact_RDW_s10/run_0001"
files = sorted(str(p.name) for p in FOLDER.glob("energy_*.pt"))
pick = w.Dropdown(options=files, value=next((f for f in files if "mlp" in f), files[-1]), description="energy")
n_valid = w.IntSlider(value=5000, min=500, max=5000, step=500, description="poses")
device = w.Dropdown(options=["mps", "cpu"] if torch.backends.mps.is_available() else ["cpu"], description="device")
display(pick, n_valid, device)

## Load the energy, the network and the draws

Re-run this cell after changing the energy or the number of poses.

In [ ]:
energy, ref, bins, info = load_energy(FOLDER / pick.value, device=device.value)
net = load_model(Path(NETWORK).expanduser(), device=device.value, weights="weights_ema").eval()
d = validation_draws(ref, n_valid.value)
pred_e = energy_predictions(energy, d)
pred_n = network_predictions(net, d)
loss = {"zero": (d["target"] ** 2).sum((1, 2)),
        "energy": ((pred_e - d["target"]) ** 2).sum((1, 2)),
        "network": ((pred_n - d["target"]) ** 2).sum((1, 2))}
k = bins.index(d["t"]).numpy()
edges = bins.edges.exp().numpy()
print(info, {n: round(float(v.mean()), 3) for n, v in loss.items()})

## Loss per bin
Mean and median per bin of the noise level s (log scale), and the share of the gap to the network that each bin carries.

In [ ]:
nb = len(edges) - 1
share = np.array([(k == i).mean() for i in range(nb)])
table = pd.DataFrame({
    "s from": [f"{edges[i]:.1e}" for i in range(nb)],
    "poses": [(k == i).sum() for i in range(nb)],
    **{f"{n} mean": [float(v[k == i].mean()) for i in range(nb)] for n, v in loss.items()},
    **{f"{n} median": [float(v[k == i].median()) for i in range(nb)] for n, v in loss.items()},
})
table["gap contribution"] = share * (table["energy mean"] - table["network mean"])
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
centres = np.sqrt(edges[1:] * edges[:-1])
for n, c in (("zero", "0.6"), ("energy", "C1"), ("network", "C0")):
    ax[0].plot(centres, table[f"{n} mean"], "o-", color=c, label=f"{n} mean")
    ax[0].plot(centres, table[f"{n} median"], "x--", color=c, alpha=0.7, label=f"{n} median")
ax[0].set(xscale="log", xlabel="s (Brownian time)", ylabel="loss", title=info)
ax[0].legend(fontsize=8)
ax[1].bar(range(1, nb + 1), table["gap contribution"])
ax[1].set(xlabel="bin", ylabel="share x (energy - network)", title=f"gap {float(loss['energy'].mean() - loss['network'].mean()):.3f}")
plt.tight_layout(); plt.show()
display(table.round(3))

## Worst poses of a bin
Pick a bin and how many poses; the table lists the poses with the largest loss of the energy.

In [ ]:
bin_pick = w.IntSlider(value=7, min=1, max=nb, description="bin")
top_n = w.IntSlider(value=10, min=3, max=50, description="worst")
out_worst = w.Output()

def show_worst(*_):
    with out_worst:
        out_worst.clear_output()
        sel = np.nonzero(k == bin_pick.value - 1)[0]
        order = sel[np.argsort(-loss["energy"][sel].numpy())][:top_n.value]
        df = pd.DataFrame({"pose": order, "t": d["t"][order].numpy(), "s": d["s"][order].numpy(),
                           **{n: v[order].numpy() for n, v in loss.items()}})
        display(df.round(3))
        fig, ax = plt.subplots(figsize=(6, 3))
        bins_h = np.logspace(0, np.log10(max(float(loss["energy"][sel].max()), 10)), 30)
        for n, c in (("energy", "C1"), ("network", "C0")):
            ax.hist(loss[n][sel].numpy(), bins=bins_h, alpha=0.5, color=c, label=n)
        ax.set(xscale="log", xlabel="loss of a pose", title=f"bin {bin_pick.value}: {len(sel)} poses"); ax.legend()
        plt.show()
        pose_pick.options = [int(i) for i in order]

pose_pick = w.Dropdown(options=[], description="pose")
bin_pick.observe(show_worst, "value"); top_n.observe(show_worst, "value")
display(w.HBox([bin_pick, top_n]), out_worst)
show_worst()

## One pose

Error per joint (energy and network), the energy's score split by feature group, the energy per joint, and the clean and noisy skeleton.

In [ ]:
out_pose = w.Output()

def show_pose(*_):
    i = pose_pick.value
    if i is None:
        return
    with out_pose:
        out_pose.clear_output()
        x, t, y = d["xt"][i], d["t"][i], d["target"][i]
        err_e = ((pred_e[i] - y) ** 2).sum(-1).numpy()
        err_n = ((pred_n[i] - y) ** 2).sum(-1).numpy()
        parts = decompose(energy, x, t)
        per_joint = energy_per_joint(energy, x, t).numpy()
        fig = plt.figure(figsize=(16, 9))
        ax = fig.add_subplot(2, 3, 1)
        pos = np.arange(19)
        ax.bar(pos - 0.2, err_e, 0.4, color="C1", label="energy")
        ax.bar(pos + 0.2, err_n, 0.4, color="C0", label="network")
        ax.set_xticks(pos, JOINTS, rotation=90, fontsize=7); ax.set(title=f"pose {i}: squared error per joint, s = {float(d['s'][i]):.1e}")
        ax.legend()
        ax = fig.add_subplot(2, 3, 2)
        names = list(parts)
        norms = np.array([parts[n].norm(dim=-1).numpy() for n in names])  # [groups, 19]
        worst = int(np.argmax(err_e))
        ax.barh(names, norms[:, worst]); ax.set(title=f"|score| by group at {JOINTS[worst]} (target {float(y[worst].norm()):.2f})")
        ax.tick_params(labelsize=7)
        ax = fig.add_subplot(2, 3, 3)
        ax.bar(pos, per_joint); ax.set_xticks(pos, JOINTS, rotation=90, fontsize=7); ax.set(title="E~ per joint")
        for col, (label, pose) in enumerate((("clean x0", d["x0"][i]), ("noisy x_t", x))):
            ax = fig.add_subplot(2, 3, 4 + col, projection="3d")
            p = points(pose).numpy()
            for a, b in enumerate(PARENTS):
                if b >= 0:
                    ax.plot(*zip(p[a], p[b]), color="k")
            ax.scatter(*p.T, s=8)
            ax.set(title=label); ax.view_init(elev=10, azim=-60)
            ax.set_box_aspect((1, 1, 1))
        ax = fig.add_subplot(2, 3, 6)
        im = ax.imshow(norms, aspect="auto", cmap="viridis")
        ax.set_yticks(range(len(names)), names, fontsize=7); ax.set_xticks(pos, JOINTS, rotation=90, fontsize=7)
        ax.set(title="|score| by group and joint"); plt.colorbar(im, ax=ax)
        plt.tight_layout(); plt.show()

pose_pick.observe(show_pose, "value")
display(pose_pick, out_pose)
show_pose()